In [0]:

storage_account_name = "adlsprimeabdel"
#CLE_D_ACCES par la clé copiée sur Azure
storage_account_key = "YOUR_KEY_HERE"

# Configuration de la connexion
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)
# Test de lecture du dossier Bronze

display(dbutils.fs.ls(f"abfss://silver@{storage_account_name}.dfs.core.windows.net/"))

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_gold = spark.read.format("delta")\
                  .option("header", True)\
                  .option("inferSchema", True)\
                  .load(f"abfss://silver@adlsprimeabdel.dfs.core.windows.net/amazon_prime_titles_silver")

df_gold.display()

In [0]:
# 1. Conversion de la colonne string en type Date
df_gold = df_gold.withColumn("date_added", try_to_date(df_gold["date_added"], "MM/dd/yyyy"))

# 2. Extraction de l'année à partir de la nouvelle colonne Date
df_gold = df_gold.withColumn("year_added", year(df_gold["date_added"]))

# 3. Affichage du résultat
df_gold.display()

In [0]:
from pyspark.sql.functions import split, trim, get

# 1. On définit la séparation de la colonne listed_in
# On le fait en une seule étape pour que Spark soit plus performant
categories_split = split(df_gold["listed_in"], ",")

# 2. On crée les colonnes en utilisant df_gold["colonne"]
# get() est indispensable pour éviter l'erreur [INVALID_ARRAY_INDEX]
df_gold = df_gold.withColumn("category_1", trim(get(categories_split, 0)))
df_gold = df_gold.withColumn("category_2", trim(get(categories_split, 1)))

# 3. Remplacement des NULLs par "Unknown" (vu sur image_baf14b.png)
df_gold = df_gold.fillna({"category_2": "Unknown"})

df_gold.display()

In [0]:
# Remplace les valeurs nulles uniquement dans la colonne spécifiée
df_gold = df_gold.fillna({"category_2": "Unknown"})

df_gold.display()

In [0]:
# Écriture du DataFrame final vers la couche Gold sur ADLS
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("path", "abfss://gold@adlsprimeabdel.dfs.core.windows.net/amazon_prime_titles_gold") \
    .save()

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS gold_layer;

In [0]:
df_gold.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_layer.prime_gold")

In [0]:
%sql
SELECT * FROM gold_layer.prime_gold